# تحليل بيانات المبيعات — Sales Data Analysis

بيانات مبيعات لشركة تبيع نماذج سيارات ومركبات مصغّرة، للفترة 2003–2005.
الهدف: تنظيف البيانات وفهمها واستخراج بعض الرؤى منها باستخدام Python.

الأدوات: pandas و matplotlib.

> يرافق المشروع داش بورد على Power BI لعرض النتائج بشكل تفاعلي.


## استيراد المكتبات

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)

# عناوين الرسوم بالإنجليزي لأن matplotlib يكسّر الحروف العربية داخل الرسم
print("done")

## فهم البيانات

أول شي نشوف شكل البيانات وحجمها.


In [ ]:
# الملف مفصول بـ ; والترميز latin1 بسبب أحرف أوروبية
df_raw = pd.read_csv('sales_data_sample1.csv', encoding='latin1', sep=';')
print(df_raw.shape)
df_raw.head(3)

الأعمدة المهمة لنا: `SALES` (قيمة المبيعة)، `PRODUCTLINE` (فئة المنتج)،
`COUNTRY`، `ORDERDATE`، `DEALSIZE`، و `CUSTOMERNAME`.
الباقي إمّا تفاصيل اتصال أو عناوين ما نحتاجها هنا.


In [ ]:
# نوع كل عمود والقيم المفقودة
pd.DataFrame({
    'type': df_raw.dtypes.astype(str),
    'missing': df_raw.isnull().sum(),
    'unique': df_raw.nunique()
})

من الجدول فوق، فيه كم مشكلة لازم نعالجها:

- عمود `Unnamed: 25` فاضي تماماً.
- `ORDERDATE` مخزّن كنص.
- `DEALSIZE` فيه قيمة غريبة `'n'`.
- بعض الأعمدة (`TERRITORY`، `STATE`، `ADDRESSLINE2`) فيها قيم مفقودة كثيرة.


## تنظيف البيانات

أشتغل على نسخة عشان ما أخرب الأصل.


In [ ]:
df = df_raw.copy()

# أعمدة ما احتجتها في التحليل: اتصال، عناوين، رمز بريدي، والعمود الفاضي
drop_cols = ['Unnamed: 25', 'ADDRESSLINE1', 'ADDRESSLINE2', 'PHONE',
             'CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'POSTALCODE']
df = df.drop(columns=drop_cols, errors='ignore')
df.shape

In [ ]:
# التاريخ نص وبتنسيقين مختلفين، نحوّله ونطلع منه السنة والشهر
df['ORDERDATE'] = pd.to_datetime(df['ORDERDATE'], format='mixed')
df['YEAR'] = df['ORDERDATE'].dt.year
df['MONTH'] = df['ORDERDATE'].dt.month
df['YEAR'].unique()

In [ ]:
# DEALSIZE فيه 32 صف قيمته 'n'، نشيلها ونبقي بس القيم الصحيحة
df['DEALSIZE'].value_counts()

In [ ]:
df = df[df['DEALSIZE'].isin(['Small', 'Medium', 'Large'])].copy()
df['DEALSIZE'].value_counts()

بالنسبة للقيم المفقودة: `TERRITORY` و `STATE` معلومات جغرافية ثانوية،
فبدل ما أحذف صفوف كاملة بسببها، ملأتها بقيمة بديلة.


In [ ]:
df['TERRITORY'] = df['TERRITORY'].fillna('Other')
df['STATE'] = df['STATE'].fillna('N/A')
df = df.drop_duplicates()

print('rows:', df.shape[0], '| missing left:', df.isnull().sum().sum())

## الإحصاء الوصفي

In [ ]:
df[['QUANTITYORDERED', 'PRICEEACH', 'SALES']].describe().round(2)

In [ ]:
print('إجمالي الإيرادات:', round(df['SALES'].sum()))
print('عدد الطلبات:', df['ORDERNUMBER'].nunique())
print('عدد العملاء:', df['CUSTOMERNAME'].nunique())
print('متوسط المبيعة:', round(df['SALES'].mean()))

ملاحظة: المتوسط أعلى من الوسيط، يعني فيه صفقات كبيرة تشدّ المتوسط لفوق.


## التحليل

ثلاثة أسئلة: أي فئة تبيع أكثر؟ أكبر الأسواق؟ ومتى تكثر المبيعات؟


### المبيعات حسب فئة المنتج

In [ ]:
product_sales = df.groupby('PRODUCTLINE')['SALES'].sum().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d9534f' if p == 'Classic Cars' else 'steelblue' for p in product_sales.index]
ax.barh(product_sales.index, product_sales.values / 1e6, color=colors)
ax.set_title('Total Revenue by Product Line')
ax.set_xlabel('Revenue (Million $)')
for i, v in enumerate(product_sales.values):
    ax.text(v/1e6 + 0.03, i, f'${v/1e6:.2f}M', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('plot1_products.png', dpi=120, bbox_inches='tight')
plt.show()

Classic Cars تتصدّر بفارق واضح، و Trains الأقل.


### أكبر الأسواق

In [ ]:
top_countries = df.groupby('COUNTRY')['SALES'].sum().sort_values().tail(10)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d9534f' if c == 'USA' else 'steelblue' for c in top_countries.index]
ax.barh(top_countries.index, top_countries.values / 1e3, color=colors)
ax.set_title('Top 10 Countries by Revenue')
ax.set_xlabel('Revenue (Thousand $)')
plt.tight_layout()
plt.savefig('plot2_countries.png', dpi=120, bbox_inches='tight')
plt.show()

usa = df[df['COUNTRY'] == 'USA']['SALES'].sum() / df['SALES'].sum() * 100
print(f'USA = {usa:.1f}% من الإيرادات')

USA أكبر سوق بفارق كبير. التركيز على سوق واحد فرصة وخطر في نفس الوقت.


### المبيعات عبر الأشهر

In [ ]:
monthly = df.groupby('MONTH')['SALES'].sum()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d9534f' if m == monthly.idxmax() else 'steelblue' for m in monthly.index]
ax.bar(monthly.index, monthly.values / 1e3, color=colors)
ax.set_title('Total Sales by Month (2003-2005)')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (Thousand $)')
ax.set_xticks(range(1, 13))
plt.tight_layout()
plt.savefig('plot3_monthly.png', dpi=120, bbox_inches='tight')
plt.show()

print(df.groupby('YEAR')['SALES'].sum().round())

نوفمبر فيه قفزة واضحة، غالباً موسم نهاية السنة.
شي ما عرفت أفسّره: أرقام 2005 أقل من 2004 — يمكن لأن البيانات تنتهي عند مايو 2005 مو لأن المبيعات نزلت فعلاً.


## الخلاصة

- أعلى فئة: Classic Cars.
- أكبر سوق: USA.
- أعلى شهر: نوفمبر.
- نمو من 2003 إلى 2004.

التوصيات المنطقية: تركيز تسويقي أكبر على Classic Cars، تجهيز مبكر لموسم نوفمبر،
والتفكير في التوسّع لأسواق غير USA لتقليل الاعتماد على سوق واحد.

العرض التفاعلي لهذه النقاط موجود في داش بورد Power BI المرفق.
